# 05 — Pack, Unpack, and Measure Reconstruction Quality

**Packing** squeezes each 8×8 float grid into one 64-bit crank word. Unpacking expands each word back into an approximate 8×8 grid.

When packing, Crankl balances three ideas roughly like this:

1. How close is the rebuilt grid to the original? (main reconstruction error)
2. Do important “shape” patterns survive compression? (persistence term)
3. Does the neighbor topology look healthy? (beta1 term)

This notebook shows both:

- **C API** pack / unpack / loss / metrics in memory
- **CLI** pack → inspect → unpack on files

Expect **some** error after round-trip — 64 floats cannot be stored perfectly in one tightly constrained word. That is compression, not a bug.

In [ ]:
from pathlib import Path
import sys
import json
import numpy as np

notebook_dir = Path.cwd() / "notebooks" if (Path.cwd() / "notebooks").exists() else Path.cwd()
sys.path.insert(0, str(notebook_dir)) if str(notebook_dir) not in sys.path else None

from crankl_demo import CranklAPI, ARTIFACT_DIR, prepare_demo_files, print_matrix, run_cli

np.set_printoptions(precision=3, suppress=True)
api = CranklAPI()
demo = prepare_demo_files()

print_matrix("Source block 0", demo.source_blocks[0])
print_matrix("Source block 1", demo.source_blocks[1])
print(f"Total floats: {demo.source_blocks.size}")
print(f"Required slots from C API: {api.lib.crankl_pack_n_slots(demo.source_blocks.size)}")

## In-memory pack → unpack round trip

128 source floats → **2** crank words. Each word unpacks to one 8×8 grid.

For each block we print: the word, the rebuilt grid, the error grid (`rebuilt − original`), and the Frobenius loss from the C API.

A small-but-nonzero error is normal.

In [ ]:
packed_words = api.pack(
    demo.source_blocks,
    persistence_weight=0.1,
    topology_weight=0.01,
)
reconstructed_blocks = api.unpack(packed_words)

for block_index, (word, source, reconstructed) in enumerate(
    zip(packed_words, demo.source_blocks, reconstructed_blocks)
):
    print(f"\nBlock {block_index} crank word: 0x{word:016x}")
    print_matrix(f"Reconstructed block {block_index}", reconstructed)
    print_matrix(f"Reconstruction error {block_index}", reconstructed - source)
    print(
        "Frobenius loss from C API:",
        f"{api.reconstruction_loss(int(word), source):.6f}",
    )

print("\nPacked-slot metrics:")
print(json.dumps(api.metrics(packed_words), indent=2))

## Same thing as files

The CLI uses the same packer and writes a `.crank` archive. `inspect --json` shows health; `unpack` writes 64 floats per slot back to a `.f32` file.

We check that CLI reconstruction matches the in-memory C API result (max absolute difference should be ~0).

In [ ]:
archive_path = ARTIFACT_DIR / "packing_example.crank"
unpacked_path = ARTIFACT_DIR / "packing_example_reconstructed.f32"

run_cli("pack", "--input", demo.source_path, "-o", archive_path)
archive_report = run_cli("inspect", archive_path, "--json", expect_json=True)
run_cli("unpack", "--input", archive_path, "-o", unpacked_path)

cli_reconstruction = np.fromfile(unpacked_path, dtype=np.float32).reshape(-1, 8, 8)
print("\nArchive report:")
print(json.dumps(archive_report, indent=2))
print("\nMax |CLI reconstruction - C API reconstruction|:",
      np.max(np.abs(cli_reconstruction - reconstructed_blocks)))

## Verify, stats, and one-shot pipeline

- **`verify`** — can this archive be read?
- **`stats`** — short header / topology summary
- **`pipeline`** — pack + turn in one command, write provenance, metrics, and an optional JSON manifest

In [ ]:
run_cli("verify", archive_path)
run_cli("stats", archive_path)

pipeline_archive = ARTIFACT_DIR / "pipeline_example.crank"
manifest_path = ARTIFACT_DIR / "pipeline_manifest.json"
run_cli(
    "pipeline", "--input", demo.source_path,
    "--target", demo.target_path,
    "--steps", 4, "--lr", 0.03,
    "-o", pipeline_archive,
    "--manifest", manifest_path,
)

manifest = json.loads(manifest_path.read_text())
print("\nPipeline manifest:")
print(json.dumps(manifest, indent=2))
print("\nPipeline archive metadata/metrics:")
run_cli("inspect", pipeline_archive, "--json")